# 先跑起来（中文运行说明）

完整说明在仓库根目录 **README.md**（打开文件夹最先看到的那一页）。

**三步：**

1. 建环境（一次）：Mac/Linux 用 `./scripts/setup_student.sh`；Windows 用 `scripts/setup_student.ps1`
2. 下数据（一次，要联网）：`python scripts/prepare.py`
3. 打开本笔记本。Kernel 选 **`yijun_emg_demo`**。保持 `FAST_DEMO = True`

英特尔 CPU + 核显：计算走 **CPU**，核显用不上。第一格打印 `DEVICE cpu` 就对了。不要装 CUDA。


# 今天我想带你走完这条线：从你的阈值，到“基础模型”这个想法

我先说清楚：这不是一节普通的机器学习课。我不会一上来就扔一个分类器给你。

你已经做过一件很具体的事——用滑动平均加一个固定阈值，让机械臂开合。
所以今天我问你的，不是“AI 能不能认肌电”，而是：

**你原来那套控制器会在哪里翻车？信号处理还能帮你到哪一步？什么时候，
我们才需要一个有可能换到下一个使用者身上还能用的 *表示*？**

编码器我选的是 **TinyMyo**（PulpBio / BioFoundation）。我会把它冻住，只训一个
线性探针。有一个文件我**不会**用：`UCI` 微调过的权重。它已经见过这些手势标签，
拿来讲“基础表示”就不干净了。

还有一句我希望你记住：如果 TinyMyo 没有打败你的手工特征，**数字怎么出来
我就怎么留**。我不会为了让故事好看去换数据集。


## 上课时你怎么跑

1. 如果还没准备过数据：`python scripts/prepare.py`（要联网；做过就跳过）
2. 下面保持 `FAST_DEMO = True`，我们课堂上用这个
3. Kernel 选 `yijun_emg_demo`，然后跟我从上到下跑

每一小节我都会先跟你讲清楚我想问什么，再让你跑代码。别急着看准确率。

你的电脑如果是英特尔处理器加核显：先看根目录 `运行说明.md`。
核显我们用不上，第一格打印 `cpu` 就对了。不要去装 CUDA。


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from IPython import get_ipython
    _ipy = get_ipython()
    if _ipy is not None:
        _ipy.run_line_magic("matplotlib", "inline")
except Exception:
    pass

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

FAST_DEMO = True
SEED = 0
MAX_ITER = 200 if FAST_DEMO else 800

import importlib
import src.dataset
import src.evaluation
import src.machine_info
import src.plotting
import src.pretrained_encoder
import src.signal_processing
for _mod in (
    src.dataset,
    src.evaluation,
    src.machine_info,
    src.plotting,
    src.pretrained_encoder,
    src.signal_processing,
):
    importlib.reload(_mod)

from src.dataset import (
    collect_raw_class_snippets, extract_binary_span, load_fast_demo,
    load_meta, load_raw_example, load_windowed, processed_paths,
    random_window_split, subject_split_indices,
)
from src.evaluation import (
    binary_accuracy, few_label_curve, fit_eval, overlapping_subjects,
    pca_2d, threshold_predict,
)
from src.plotting import (
    plot_confusion, plot_crosstalk_diagram, plot_few_label_curves,
    plot_grouped_bars, plot_model_ab_table, plot_notch_before_after,
    plot_pca_scatter, plot_raw_channels, plot_raw_class_gallery,
    plot_summary_progression,
    plot_collection_photos, plot_four_dataset_channels,
    plot_three_pipelines, plot_tinymyo_mask_story,
    plot_threshold_controller, plot_threshold_sweep,
    plot_moving_average_as_lowpass, plot_time_fft_psd,
    plot_window_length_sweep, set_plot_language,
)
from src.machine_info import format_machine_report
from src.pretrained_encoder import (
    build_encoder, count_parameters, default_checkpoint_path,
    encode_windows, load_pretrained_weights, select_device,
)
from src.signal_processing import (
    add_synthetic_mains, fft_magnitude, mean_channel_envelope,
    notch_filter, peak_near, welch_psd,
)

font_name = set_plot_language("zh")
DEVICE = select_device()
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ROOT", ROOT)
print("FAST_DEMO", FAST_DEMO)
print("DEVICE", DEVICE)
print("中文字体", font_name)
print(format_machine_report(lang="zh"))


In [ ]:
CLASS_NAMES_ZH = {
    0: "放松",
    1: "握拳",
    2: "腕屈曲",
    3: "腕伸展",
    4: "桡偏",
    5: "尺偏",
}

meta = load_meta()
rec = load_raw_example(processed_paths()["example"])
paths = processed_paths()
if FAST_DEMO:
    demo = load_fast_demo()
    X, y, subject = demo["X"], demo["y"].astype(np.int32), demo["subject"].astype(np.int32)
    Z, F = demo["embeddings"], demo["features"]
    train_subjects = [int(s) for s in demo["train_subjects"]]
    test_subjects = [int(s) for s in demo["test_subjects"]]
    window_samples = int(demo["window_samples"])
else:
    windows = load_windowed(paths["windows"])
    X, y, subject = windows.X, windows.y, windows.subject
    Z = np.load(paths["embeddings"])
    F = np.load(ROOT / "cache" / "handcrafted_features.npz")["features"]
    train_subjects = meta["splits"]["train"]
    test_subjects = meta["splits"]["test"]
    window_samples = int(windows.window_samples)

fs = float(meta["uci_fs"])
pair = meta["binary_pair"]
if pair["kind"] == "rest_vs_fist":
    pair_title = "手部放松 vs 握拳"
else:
    pair_title = f"基于幅度的二分类：{pair['low_name']} vs {pair['high_name']}"
class_names = CLASS_NAMES_ZH
print("窗口协议:", meta["window_protocol"])
print("X", X.shape, "y", y.shape, "受试者数", len(np.unique(subject)))
print("预训练嵌入 Z", Z.shape, "手工特征 F", F.shape)
print("二分类演示:", pair_title)


---
# 数据集 — 桌上其实有四份，通道对的不是同一块肉

分类用的是 **UCI 481**：Myo，8 通道，200 Hz，俄罗斯罗巴切夫斯基大学。
TinyMyo 的编码器是在另外三份 **2 kHz** 数据上预训练的：NinaPro **DB6**、
**DB7**、**EMG2Pose**。都叫肌电，**电极几何不是同一回事**。

- **UCI / Myo：** 前臂等间距 8 个舱。官方**没有**“第几通道=哪块肌肉”。
  Lobov 2018 只说戴正时 2/4/6/8 大致靠近桡侧腕屈、伸肌群、尺侧腕伸、尺侧腕屈。
- **DB6：** 14 个 Delsys。近端桡肱关节一圈 8 个，再往下 6 个。文件常有 16 列，2 列空。
- **DB7：** 12 个 Delsys。8 个一圈，加上指浅屈/指总伸，再加上肱二头、肱三头。
- **EMG2Pose：** 腕带 16 个双极舱，尺骨上方留缝，**0 和 15 贴在缝两边**。

TinyMyo 按最多 16 通道来。UCI 只有 8 路，后面是空位。探针没赢，这是原因之一。

照片来自 NinaPro 官网（采集流程 + DB6 真人手）。UCI 论文原图被出版社拦住了，
我不会假装那是“UCI 受试者本人”。


In [ ]:
fig = plot_collection_photos(ROOT / "outputs" / "datasets_collection_photos.png")
display(fig)
plt.show()
fig = plot_four_dataset_channels(ROOT / "outputs" / "datasets_channel_maps.png")
display(fig)
plt.show()


---
# Foundation model — 我先把 TinyMyo 的作业讲清楚

它不是更大的分类器。预训练**没有**今天这六类 UCI 标签。
吃的是上表那三份 2 kHz 数据。权重：`TinyMyo.safetensors`。UCI 微调那份不用。

原始信号：最多 16 通道 × 1000 点（0.5 秒）。盖住大约一半 10 ms 的小段，
再把那 20 个电压填回去。训完重建头丢掉。我们冻住编码器，只问手势好不好切。

通道对不齐、采样率差十倍，它都可以输给你写的 32 个数。这不是事故，是今天的课。


In [ ]:
fig_mask = plot_tinymyo_mask_story(ROOT / "outputs" / "exp10_mask_story.png")
display(fig_mask)
plt.show()


---
# 第一幕 — 我原来会怎么做

换作是我，拿到肌电我第一反应也是：幅度、平均、滤波。
**我能不能把问题“洗干净”？** 我们一起试。


## 实验 1 — 你先看看原始 EMG 长什么样

我想让你先回答一个很朴素的问题：只盯着幅度看，你能不能判断她想干什么？

表面肌电不是一个数。它是贴在皮肤上、随时间跳的**多通道**电压。
每个电极看到的，都是附近几块肌肉搅在一起的混合物。

数据是 UCI EMG gestures（id 481）。TinyMyo 的脚本按 **200 Hz** 来用。
官方编码我列在这里，你对照着看图：

| 编码 | 含义 |
| --- | --- |
| 0 | 未标记 |
| 1 | 手部放松 |
| 2 | 握拳 |
| 3 | 腕屈曲 |
| 4 | 腕伸展 |
| 5 | 桡偏 |
| 6 | 尺偏 |
| 7 | 伸掌（不是所有人都做了） |

请你现在就记住一句：**未标记不是放松。** 放松是 1。0 只是“这段没标”。

我先给你看一段连续记录，再按 0–7 **各切一段保持**分别画出来。
伸掌不是人人都做。这份示例文件里如果没有 7，我会去受试者 11、30 那里借一段——
他们是官方记录里做过伸掌的人。标题里我会写清换了人，不会假装还是同一个人。


In [ ]:
from src.dataset import collect_raw_class_snippets
from src.plotting import plot_raw_class_gallery

t = rec.time_ms / 1000.0
sl = slice(0, int(min(len(t), 8 * fs)))
print("示例文件:", rec.source_path)
print("受试者", rec.subject, "形状 (时间, 通道)", rec.emg.shape, "采样率", rec.fs)
print("这份文件里的原始标签:", sorted(int(x) for x in np.unique(rec.raw_label)))
print("分类窗口 X", X.shape, "类别数", len(np.unique(y)), "（分类阶段已丢掉未标记和伸掌）")
fig = plot_raw_channels(
    t[sl], rec.emg[sl], rec.raw_label[sl], channels=(0, 1, 2, 3),
    title=f"连续记录（前几秒）— 受试者 {rec.subject}",
)
plt.show()

RAW_NAMES_ZH = {
    0: "未标记",
    1: "手部放松",
    2: "握拳",
    3: "腕屈曲",
    4: "腕伸展",
    5: "桡偏",
    6: "尺偏",
    7: "伸掌",
}
snippets = collect_raw_class_snippets(rec, labels=range(8), max_seconds=3.0)
print("分别找到的编码:", sorted(snippets))
for code in range(8):
    snip = snippets.get(code)
    if snip is None:
        print(f"  {code} {RAW_NAMES_ZH[code]}: 没有找到")
        continue
    src = "本文件" if snip["from_primary"] else f"受试者 {snip['subject']}"
    print(f"  {code} {RAW_NAMES_ZH[code]}: {snip['n_samples']} 点  来自 {src}")
fig = plot_raw_class_gallery(
    snippets, RAW_NAMES_ZH, channels=(0, 1, 2, 3),
    title="官方 0–7：每个编码一段保持（通道 1–4）",
)
plt.show()


你看完图，我想听你说：如果两种动作的幅度差不多高，你那个固定阈值还能稳吗？


## 实验 2 — 我把你原来的阈值控制器复原给你看

整流，再滑动平均，再画一条线：这不就是你原来的开关吗？
我现在就问：**它能不能做成开/关控制器？**

整流是把负电压折上来。滑动平均是在估一条慢的*包络*。阈值就是检测器：
低激活一边，高激活一边。

我不会先假定 UCI 的 1 和 2 一定就是“放松对握拳”。`prepare.py` 已经对过官方
标签，也算过真实 RMS。配对写在 `meta["binary_pair"]` 里。
要是放松并不明显更安静，我会改口叫“基于幅度的二分类”，不会硬叫放松 vs 握拳。


In [ ]:
print("配对类型:", pair["kind"])
print("低激活:", pair["low_name"], "RMS", f"{pair['low_rms']:.3e}")
print("高激活:", pair["high_name"], "RMS", f"{pair['high_rms']:.3e}")
time_s, emg, raw_lab, truth = extract_binary_span(
    rec, pair["low_raw_label"], pair["high_raw_label"]
)
env = mean_channel_envelope(emg, win=20, method="mav")
thr = 0.5 * (env[truth == 0].mean() + env[truth == 1].mean())
pred = threshold_predict(env, thr)
acc = binary_accuracy(pred, truth)
print(f"所选阈值 {thr:.4g}   二分类准确率 {acc:.3f}")
fig = plot_threshold_controller(
    time_s, emg[:, 0], env, thr, pred, truth,
    title=f"{pair_title}  （准确率={acc:.2f}）",
)
plt.show()


In [ ]:
thrs = np.linspace(float(env.min()), float(env.max()), 25)
accs = [binary_accuracy(threshold_predict(env, t_), truth) for t_ in thrs]
fig = plot_threshold_sweep(thrs, accs, title="同一段记录，不同阈值")
plt.show()

wins = [5, 10, 20, 40, 80]
wacc = []
for w in wins:
    e = mean_channel_envelope(emg, win=w, method="mav")
    t_ = 0.5 * (e[truth == 0].mean() + e[truth == 1].mean())
    wacc.append(binary_accuracy(threshold_predict(e, t_), truth))
fig = plot_window_length_sweep(wins, wacc, title="准确率 vs 滑动平均窗口长度")
plt.show()
print("窗口长度", wins)
print("准确率   ", [round(a, 3) for a in wacc])

fig = plot_moving_average_as_lowpass(
    time_s, emg[:, 0], windows=[5, 20, 80], fs=fs, truth=truth,
    title="滑动平均在做什么：同一段整流信号，三种窗口",
)
plt.show()


我请你盯三件事。

第一张三行图：上面是原始信号、包络和那条红线；中间是控制器的 0/1；下面是
真实标签。中间和下面有多像，就是这个开关有多像样。

第二张是我在同一段记录上扫不同阈值。你会看到：线画高一点、低一点，数字会变。
所以阈值是调出来的，不是物理常数。

第三张是窗口长度。后面我还多画了一张——同一段整流信号叠 5 / 20 / 80 点平均，
再画均匀窗口的 sinc 幅频响应。我想让你亲眼看见：**平均就是低通**。
窗口越长，边沿越钝、越滞后，通带也越窄。

平滑不是免费的。你赢得的是好看的包络，付出的是响应时间。
这段记录上最好的阈值，换电极、出汗、换一个人，我不敢保证还最好。

你能不能告诉我：窗口变大，为什么更平滑？手已经握上了，开关会怎样？


## 实验 3 — 幅度看不清的时候，我带你去频域

时域一团乱的时候，频谱能不能告诉我：这是*哪一类*干扰？

不同的物理过程占不同的频率。仪器干扰可以窄得像一根针。肌电、还有很多
生理性串扰，往往是宽带的。

我先画**观测到的**频谱。这张图上如果没有 50 Hz 尖峰，我就不对你说
“你看，这就是市电”。没有证据，我不声称这份数据有 50 Hz 伪迹。

然后我会故意加一个正弦，并且写明 **合成**。那张图才是我用来教陷波的，
不是这份数据里本来就有的东西。


In [ ]:
high_idx = np.where(raw_lab == pair["high_raw_label"])[0]
a = int(high_idx[0]); z = min(len(emg), a + int(2 * fs))
x = emg[a:z, 0]
tt = np.arange(len(x)) / fs
freq, mag = fft_magnitude(x, fs)
pf, psd = welch_psd(x, fs)
print("观测窗口、50 Hz 附近的 FFT 幅度:", round(peak_near(freq, mag, 50), 6))
fig = plot_time_fft_psd(
    tt, x, freq, mag, pf, psd,
    title="观测窗口 — 不声称存在 50 Hz 伪迹",
)
plt.show()


In [ ]:
x_syn = add_synthetic_mains(x, fs, freq=50.0)
f2, m2 = fft_magnitude(x_syn, fs)
p2f, p2 = welch_psd(x_syn, fs)
fig = plot_time_fft_psd(
    tt, x_syn, f2, m2, p2f, p2,
    title="合成教学例子：x_真实(t) + A sin(2π·50·t)",
    mark_hz=50.0, mark_label="合成 50 Hz",
)
plt.show()


如果干扰占的是很窄的一条，你会选哪一类滤波器？想完再往下跑。


## 实验 4 — 滤波能帮你，但帮不到底

我能把问题滤干净吗？短短几分钟，我们把边界摸清楚。

当有用的和讨厌的在频谱上能分开，滤波很爽。可邻近那块肌肉的串扰，常常
和你要的肌电挤在**同一段频率**里。那时候不存在一把完美的剪刀。


In [ ]:
x_notch = notch_filter(x_syn, fs, freq=50.0)
pf_b, psd_b = welch_psd(x_syn, fs)
pf_a, psd_a = welch_psd(x_notch, fs)
fig = plot_notch_before_after(
    tt, x_syn, x_notch, pf_b, psd_b, pf_a, psd_a,
    title="好消息：合成的窄带谱线可以被陷波去掉",
)
plt.show()
fig = plot_crosstalk_diagram()
plt.show()


合成 50 Hz 上，陷波是成功的。频率分开了，线就能剪掉。

现在我换一个假设，请你看这个式子：

\[
\text{目标肌电频谱} \;\cap\; \text{串扰频谱} \;\neq\; \emptyset
\]

两块肌肉的频段叠在一起时，你告诉我：哪个截止频率能把它们完美切开？

我的答案是——**可能没有。** 滤波到此为止。问题从“把信号分开”变成
“从模式里认出来”。


---
# 第二幕 — 那我让机器去认模式

与其幻想完美隔离每一块肌肉，我换个问法：多通道搅在一起的*样子*里，
有没有能用的信息？


## 实验 5 — 先用你能解释的手工特征

我不再问“过没过线”，我改问：“这一组测量值的组合，是不是对应这个动作？”

RMS、平均绝对值、波形长度、过零次数——这些是工程师给人看的摘要。
8 通道 × 4 个数，每个窗变成一个 32 维向量。后面的模型**只吃这 32 个数**，
不再看原始电压曲线。

先把名字说死，免得你和“线性回归”搅在一起：

**这不是**最小二乘线性回归。那个是预测一个连续数字（比如力有多大）。
代码里是 `StandardScaler` + `LogisticRegression`，叫**多项逻辑回归**，
也就是带 softmax 的**线性分类器**。

对每个手势 \(k\)（放松、握拳、腕屈……一共 6 类），它学一条直线分数：

$$
s_k = w_k\cdot \tilde{x} + b_k
$$

\(\tilde{x}\) 是标准化之后的 32 个数：每个特征减训练集均值、除标准差。
不这一步，RMS 和过零次数差几个数量级，权重没法比。

六个分数进 softmax，变成加起来为 1 的概率；预测取 \(s_k\) 最大的那一类。
它要学的就是 6 组 \((w_k, b_k)\)，大约 \(6\times 32 + 6 = 198\) 个参数。
**没有隐藏层，没有神经网络。** 和你实验 2 的阈值比：阈值是一个数过线；
这里是 32 维空间里 6 张超平面。所以它能分六类，不只是放松对握拳。

我**故意**先做随机窗口划分。数字往往会很好看。然后我再按人切开。
你先别被第一组数字骗到。


In [ ]:
names = [class_names[i] for i in sorted(np.unique(y))]
rand_tr, rand_te = random_window_split(y, test_size=0.25, seed=SEED)
print("随机划分 训练/测试窗口数", len(rand_tr), len(rand_te))
print("训练和测试里重叠的受试者:", sorted(int(s) for s in overlapping_subjects(subject, rand_tr, rand_te)))
rand_f = fit_eval(F[rand_tr], y[rand_tr], F[rand_te], y[rand_te], seed=SEED, max_iter=MAX_ITER)
W = rand_f["clf"].named_steps["clf"].coef_
b = rand_f["clf"].named_steps["clf"].intercept_
print("每个窗的手工特征维数", F.shape[1], "（8 通道 × 4）")
print("权重矩阵 W 形状", W.shape, "（类别 × 特征）; 偏置", b.shape)
print("可训练参数", rand_f["n_probe_params"], "（W 的元素 + 偏置）")
print("预测规则: 对标准化后的 x，算 6 个分数 s = W x + b，取 argmax")
print("手工特征 + 线性分类，随机划分准确率:", round(rand_f["accuracy"], 3))
fig = plot_confusion(rand_f["confusion"], names, f"随机窗口划分  准确率={rand_f['accuracy']:.3f}")
plt.show()


请你看训练和测试里重叠的受试者名单。同一个人两边都有的话，97%
**没有**证明这套东西能用在下一个没见过的人身上。

一个基准数字，只有你知道“到底扣住了什么”，它才有意义。


In [ ]:
sub_tr, sub_te = subject_split_indices(subject, train_subjects, test_subjects)
print("训练受试者", train_subjects)
print("测试受试者", test_subjects)
print("训练窗口", len(sub_tr), "测试窗口", len(sub_te))
if len(sub_tr) == 0 or len(sub_te) == 0:
    print("这个 FAST_DEMO 子集里，留出受试者不够。")
    sub_f = None
else:
    sub_f = fit_eval(F[sub_tr], y[sub_tr], F[sub_te], y[sub_te], seed=SEED, max_iter=MAX_ITER)
    print("手工特征 + 线性分类，按人划分准确率:", round(sub_f["accuracy"], 3))
    fig = plot_confusion(sub_f["confusion"], names, f"按人划分  准确率={sub_f['accuracy']:.3f}")
    plt.show()


机器在这里学到了你的固定阈值没学到的什么？我想听你用自己的话讲。


## 实验 6 — 我冻住 TinyMyo，只问表示好不好用

在看到*这些* UCI 标签之前，TinyMyo 的预训练有没有已经给出一个
让手势差不多能被一条直线切开的空间？

听清：这**不是微调**。编码器我冻住。我只在嵌入上训线性分类器。

我用的权重：`pretraining/TinyMyo/TinyMyo.safetensors`  
我不用的：`UCI_EMG/UCI_finetune_5sec.safetensors`


In [ ]:
ckpt = default_checkpoint_path()
model = build_encoder(img_size=window_samples, in_chans=8, device=DEVICE)
load_report = load_pretrained_weights(model, ckpt)
model.to(DEVICE)
params = count_parameters(model)
print("权重文件", ckpt)
print("DEVICE", DEVICE, "| 编码器参数所在设备", next(model.parameters()).device)
print("缺失键（若头不同，可以预期）:", load_report["missing_keys"][:8])
print("编码器参数:", params)
print(f"原始输入形状  [{len(X)}, 8, {window_samples}]")
print(f"冻结编码器 → 嵌入形状 {tuple(Z.shape)}  （维数={Z.shape[1]}）")
smoke = encode_windows(model, X[:1], batch_size=1, device=DEVICE)
print("在", DEVICE, "上编码一个窗口 →", smoke.shape)
print("课堂数字仍用缓存嵌入；编码器没有在训练。")


我想让你看清楚坐标的意义变了。

原始的 \(X\) 是一张电压表：某一电极、某一时刻。

TinyMyo 给你的是

$$
z = [z_1, z_2, \ldots, z_{1536}]
$$

一般是 `8 × 192`。这些数**不再对应**某个采样点。它们是预训练
（掩码 patch 重建）觉得有用的时间结构和跨通道结构，不是为我今天的
手势标签专门学的。

如果线性分类器在 \(z\) 上好用，只说明：和运动有关的信息，在这个
表示里变得更好抽了。这就是线性探针在问的事。


In [ ]:
rand_z = fit_eval(Z[rand_tr], y[rand_tr], Z[rand_te], y[rand_te], seed=SEED, max_iter=MAX_ITER)
print("可训练的探针参数", rand_z["n_probe_params"])
print("冻结的编码器参数 ", params["frozen"])
print("线性探针，随机划分准确率:", round(rand_z["accuracy"], 3))
if sub_f is not None:
    sub_z = fit_eval(Z[sub_tr], y[sub_tr], Z[sub_te], y[sub_te], seed=SEED, max_iter=MAX_ITER)
    print("线性探针，按人划分准确率:", round(sub_z["accuracy"], 3))
    fig = plot_confusion(sub_z["confusion"], names, f"TinyMyo 探针，按人划分 准确率={sub_z['accuracy']:.3f}")
    plt.show()
else:
    sub_z = None


编码器没有被训练来解这个分类作业。我问的只是：它预训练时学到的空间，
是不是已经让后面的类别比较好分开。


## 实验 7 — 我用 PCA 让你看一眼，不是用 t-SNE 变魔术

哪一种表示看起来更好切？

PCA 画得好看，**不能**当证据。真正比的是分类数字。
PCA 只是线性的直觉图，我把它放在这里是为了让你形成画面，不是为了下结论。


In [ ]:
pca_f, ev_f = pca_2d(F, seed=SEED)
pca_z, ev_z = pca_2d(Z, seed=SEED)
print("手工特征 PCA 方差比", np.round(ev_f, 3))
print("TinyMyo     PCA 方差比", np.round(ev_z, 3))
note = "只是视觉直觉；真正比较看分类数字。"
fig = plot_pca_scatter(pca_f, y, class_names, subject, title="手工特征的 PCA", note=note)
plt.show()
fig = plot_pca_scatter(pca_z, y, class_names, subject, title="冻结 TinyMyo 嵌入的 PCA", note=note)
plt.show()


你觉得哪边更好切？说完再去对数字。数字不同意图的时候，听数字。


---
# 第三幕 — 它真的能换人用吗？

实验室里会分类，还不等于假肢里能用。这一幕我希望你坐直一点。


## 实验 8 — 标签很少的时候

两边都只给你很少标注，预训练还会不会更值钱？

这是我带你做的第一次**数据效率**检查。有用。但还不是今天最难的那一问。


In [ ]:
train_mask = np.isin(subject, train_subjects)
test_mask = np.isin(subject, test_subjects)
if test_mask.sum() == 0:
    idx_local = np.where(train_mask)[0]
    tr_i, te_i = random_window_split(y[train_mask], test_size=0.3, seed=SEED)
    tr_idx, te_idx = idx_local[tr_i], idx_local[te_i]
    few_note = "训练和测试是同一批人（只改变标签数量）"
else:
    tr_idx, te_idx = np.where(train_mask)[0], np.where(test_mask)[0]
    few_note = "用见过的人的窗口训练；在没见过的人上测试"
print(few_note)
print("训练窗口", len(tr_idx), "测试窗口", len(te_idx))
fractions = [1.0, 0.25, 0.10]
if len(tr_idx) >= 80:
    fractions.append(0.05)
seeds = [0, 1] if FAST_DEMO else [0, 1, 2]
curve_f = few_label_curve(F[tr_idx], y[tr_idx], F[te_idx], y[te_idx], fractions, seeds, max_iter=MAX_ITER)
curve_z = few_label_curve(Z[tr_idx], y[tr_idx], Z[te_idx], y[te_idx], fractions, seeds, max_iter=MAX_ITER)
for name, curve in [("手工特征", curve_f), ("TinyMyo 探针", curve_z)]:
    print(name)
    for frac, stats in curve.items():
        print(f"  {100*frac:3.0f}% 标签  {stats['mean']:.3f} ± {stats['std']:.3f}")
fig = plot_few_label_curves(
    {"手工特征 + 线性分类": curve_f, "冻结 TinyMyo + 线性探针": curve_z},
    title=f"有限标注 — {few_note}",
)
plt.show()


有用，对吧？但还不是最难的。

如果基础模型探针**没有赢**，我可以跟你一起列原因：UCI 这任务本身不狠；
预训练是 2 kHz、16 通道，我们下游是 200 Hz、8 通道；预处理也不一样；
线性探针动不了编码器；静态手势上手工特征本来就强。

我不会为了让它赢去换数据集。输了就摆在桌上。


## 实验 9 — 留出一个人（我把高潮放在这里）

手势名字没变，人一换，分布为什么就变了？

$$
\boxed{\text{在见过的人身上训练}}
\quad\rightarrow\quad
\boxed{\text{在从未见过的人身上测试}}
$$

如果你要做假肢控制器，这才是更像现实的考试。解剖、募集、电极位置、
皮肤阻抗、记录条件、串扰，全都会变。标签还是“握拳”，信号已经不是
同一回事了。


In [ ]:
if sub_f is None or sub_z is None:
    print("这个子集做不了留出受试者实验。")
    print("这是数据限制，不是藏起来的结果。")
else:
    print("手工特征，未见过的人:", round(sub_f["accuracy"], 3))
    print("TinyMyo 探针，未见过的人:", round(sub_z["accuracy"], 3))
    fig = plot_grouped_bars(
        ["留出受试者"],
        {"手工特征": [sub_f["accuracy"]], "TinyMyo 探针": [sub_z["accuracy"]]},
        ylabel="准确率",
        title="在见过的人上训练，在没见过的人上测试",
    )
    plt.show()


为什么这对假肢，比“把同一个人的窗口随机切开”难得多？
你要是能讲清楚这一点，今天这节课就值了。


## 实验 10 — 我为什么要跟你提基础模型，它又不是魔法

基础模型不是“更大的分类器”换了个名字。

我想让你带走的变化是：

**以前：每个任务从头学。现在：先学一个还能再用的表示。**

它不会魔法般去掉串扰。真正的研究问题是：它能不能学到对运动有用、
换使用者和换记录条件之后还站得住的结构。

### 他们预训练时到底在干什么：盖住一段，再把电压填回去

今天**没有**重新训 TinyMyo。`TinyMyo.safetensors` 是现成的编码器。
预训练也**不是**在你眼前这份 UCI 481 上做的——那是 NinaPro DB6/DB7、EMG2Pose
那种 2 kHz 的大数据。作业长什么样，以 `vendor/TinyMyo.py` 和论文为准。

**原始信号**是一窗时域电压：最多 16 通道、1000 个点（2 kHz 下 0.5 秒）。
先做 20–450 Hz 带通、50 Hz notch，再缩到 \([-1,1]\)。**没有手势标签。**
它不是在学“这是不是握拳”。

**切法：** 每个通道单独切成长 20、步长 20 的小段。2 kHz 下 20 点 = 10 ms。
16 通道就是 800 个 token。每一小段先变成 192 个数，再进 Transformer。

**怎么盖：** 随机盖住大约一半的小段，整段嵌入换成一个可学习的 `[MASK]`。
盖掉的是**几十毫秒的波形**，不是一个采样点，所以它不能靠简单插值作弊，
得看左右时间和别的电极。

**复原的是什么：** 预训练头只是一层线性：每个 token 吐回 20 个电压，去对
那一小段原始波形。损失是 Smooth L1，被盖住的一半要认真还原，看得见的一半
也扣一点（系数 0.1），免得只抄输入。训完这个头丢掉。我们冻住的是中间的
编码器；线性探针只问手势好不好切。

第一张图是我画的**示意图**，灰色条就是被换成 `[MASK]` 的 patch。
那不是 TinyMyo 对 UCI 跑出来的真重建。


In [ ]:
fig = plot_three_pipelines()
plt.show()
fig = plot_model_ab_table()
plt.show()
fig = plot_summary_progression(ROOT / "outputs" / "emg_to_foundation_model_zh.png")
plt.show()
print("已保存", ROOT / "outputs" / "emg_to_foundation_model_zh.png")


### 电池供电的假肢里面，你会放哪一个？

我没有标准答案。你要同时想：准不准、稳不稳、换人行不行、延迟、内存、
功耗、可不可靠、校准烦不烦。

**榜上最好的模型，不必是你手里最好的工程系统。**

我今天想留给你的工程结论就一句：实验室里好用的分类器，并不自动等于
真实世界里好用的假肢控制器。
